In [1]:
import warnings
import pandas as pd

In [37]:
data = {
    "event_id": ['e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9'],
    "case:concept:name": [1, 1, 1, 1, 2, 2, 2, 3, 3],
    "time:timestamp": pd.to_datetime([
        '2024-01-01 08:00', '2024-01-01 08:15', '2024-01-01 09:00', '2024-01-01 09:30',
        '2024-01-01 08:05', '2024-01-01 08:20', '2024-01-01 10:00',
        '2024-01-02 07:00', '2024-01-02 07:45'
    ]),
    "concept:name":[
        'Register', 'Check', 'Check', 'Approve',
        'Register', 'Approve', 'Close',
        'Register', 'Check'
    ]
}

df = pd.DataFrame(data)
df

,event_id,case:concept:name,time:timestamp,concept:name
0,e1,1,2024-01-01 08:00:00,Register
1,e2,1,2024-01-01 08:15:00,Check
2,e3,1,2024-01-01 09:00:00,Check
3,e4,1,2024-01-01 09:30:00,Approve
4,e5,2,2024-01-01 08:05:00,Register
5,e6,2,2024-01-01 08:20:00,Approve
6,e7,2,2024-01-01 10:00:00,Close
7,e8,3,2024-01-02 07:00:00,Register
8,e9,3,2024-01-02 07:45:00,Check


In [38]:
dt = pd.to_datetime("2024-01-01 08:00")
dt
#type(dt)

Timestamp('2024-01-01 08:00:00')

In [39]:
df = df.sort_values(["time:timestamp"])
df

,event_id,case:concept:name,time:timestamp,concept:name
0,e1,1,2024-01-01 08:00:00,Register
4,e5,2,2024-01-01 08:05:00,Register
1,e2,1,2024-01-01 08:15:00,Check
5,e6,2,2024-01-01 08:20:00,Approve
2,e3,1,2024-01-01 09:00:00,Check
3,e4,1,2024-01-01 09:30:00,Approve
6,e7,2,2024-01-01 10:00:00,Close
7,e8,3,2024-01-02 07:00:00,Register
8,e9,3,2024-01-02 07:45:00,Check


In [40]:
activity_dummies = pd.get_dummies(df['concept:name'], prefix='count')
activity_dummies
#type(activity_dummies)

,count_Approve,count_Check,count_Close,count_Register
0,False,False,False,True
4,False,False,False,True
1,False,True,False,False
5,True,False,False,False
2,False,True,False,False
3,True,False,False,False
6,False,False,True,False
7,False,False,False,True
8,False,True,False,False


In [41]:
c_idx_series = df['case:concept:name']
c_idx_series

0    1
4    2
1    1
5    2
2    1
3    1
6    2
7    3
8    3
Name: case:concept:name, dtype: int64

In [42]:
activity_counts = activity_dummies.groupby(c_idx_series).cumsum()
activity_counts

,count_Approve,count_Check,count_Close,count_Register
0,0,0,0,1
4,0,0,0,1
1,0,1,0,1
5,1,0,0,1
2,0,2,0,1
3,1,2,0,1
6,1,0,1,1
7,0,0,0,1
8,0,1,0,1


In [43]:
df = pd.concat([df, activity_counts], axis=1)
df

,event_id,case:concept:name,time:timestamp,concept:name,count_Approve,count_Check,count_Close,count_Register
0,e1,1,2024-01-01 08:00:00,Register,0,0,0,1
4,e5,2,2024-01-01 08:05:00,Register,0,0,0,1
1,e2,1,2024-01-01 08:15:00,Check,0,1,0,1
5,e6,2,2024-01-01 08:20:00,Approve,1,0,0,1
2,e3,1,2024-01-01 09:00:00,Check,0,2,0,1
3,e4,1,2024-01-01 09:30:00,Approve,1,2,0,1
6,e7,2,2024-01-01 10:00:00,Close,1,0,1,1
7,e8,3,2024-01-02 07:00:00,Register,0,0,0,1
8,e9,3,2024-01-02 07:45:00,Check,0,1,0,1


In [44]:
# Direcctly follows counts

In [47]:
df['prev_activity'] = df.groupby('case:concept:name')['concept:name'].shift(1)
df

,event_id,case:concept:name,time:timestamp,concept:name,count_Approve,count_Check,count_Close,count_Register,prev_activity
0,e1,1,2024-01-01 08:00:00,Register,0,0,0,1,NaN
4,e5,2,2024-01-01 08:05:00,Register,0,0,0,1,NaN
1,e2,1,2024-01-01 08:15:00,Check,0,1,0,1,Register
5,e6,2,2024-01-01 08:20:00,Approve,1,0,0,1,Register
2,e3,1,2024-01-01 09:00:00,Check,0,2,0,1,Check
3,e4,1,2024-01-01 09:30:00,Approve,1,2,0,1,Check
6,e7,2,2024-01-01 10:00:00,Close,1,0,1,1,Approve
7,e8,3,2024-01-02 07:00:00,Register,0,0,0,1,NaN
8,e9,3,2024-01-02 07:45:00,Check,0,1,0,1,Register
